# Brain Tumor MRI Classification using PyTorch

---
#### DISCLAIMER: THIS IS NOT MEDICAL ADVICE, DO NOT SELF DIAGNOSE

## Imports and loading data

In [1]:
import torch
from torch import nn, optim
from torch.nn import Flatten
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

# use gpu if it's available if not cpu which is much slower
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Image transformations
Certain transformations are needed before using an image, e.g resize, changing to tensor and normalizing

In [2]:
# Every image will go through this transformation pipeline
pic_trans = [
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
]

In [3]:
train_dl = DataLoader(
    datasets.ImageFolder('data/Training', pic_trans),
    batch_size=32,
    shuffle=True,
    num_workers=4, # speeds up loading data
    pin_memory=True
)

test_dl = DataLoader(
    datasets.ImageFolder('data/Testing', pic_trans),
    batch_size=32,
    shuffle=False, # there is no need to shuffle, as this is just for evaluation
    num_workers=4, # speeds up loading data
    pin_memory=True
)

## Define model

In [5]:
model = nn.Sequential(
    nn.Conv2d(3, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(128 * 16 * 16, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 4) # final layer, which takes 256 as input and 4 as output (4 classes
).to(device)

In [6]:
opt = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()